In [5]:
import pandas as pd
import altair as alt
from ecostyles import EcoStyles  # adjust import to match your package

styles = EcoStyles()
styles.register_and_enable_theme()

# Colour comes from the theme palette via a colour encoding, with no legend
COLOUR = alt.Color("series:N", legend=None)


def bar_with_labels(df, x, y, title, subtitle, fmt=".1f"):
    df = df.assign(series="main")
    base = alt.Chart(df)
    bars = base.mark_bar().encode(
        x=alt.X(f"{x}:N", title=None, axis=alt.Axis(labelAngle=0)),
        y=alt.Y(f"{y}:Q", title="£m"),
        color=COLOUR,
    )
    labels = base.mark_text(dy=-6).encode(
        x=f"{x}:N", y=f"{y}:Q", text=alt.Text(f"{y}:Q", format=fmt)
    )
    return (bars + labels).properties(title=alt.Title(title, subtitle=subtitle))


# 1. Cumberland budget gap
gap = pd.DataFrame({"year": ["2027-28", "2028-29", "2029-30"],
                    "gap": [25.57, 50.85, 59.51]})
c1 = bar_with_labels(gap, "year", "gap",
    "Cumberland Council projected budget gap",
    "£m, medium-term financial plan. Source: CIPFA external assurance review (May 2026)")

# 2. Westmorland and Furness financing costs
wf = pd.DataFrame({"year": ["2026/27", "2027/28", "2028/29", "2029/30", "2030/31"],
                   "ratio": [4.02, 5.30, 6.08, 6.44, 6.78],
                   "series": "main"})
c2 = alt.Chart(wf).mark_line(strokeWidth=1.8, point=True).encode(
    x=alt.X("year:N", title=None, axis=alt.Axis(labelAngle=0)),
    y=alt.Y("ratio:Q", title="% of net revenue", scale=alt.Scale(domain=[0, 8])),
    color=COLOUR,
).properties(title=alt.Title(
    "Westmorland and Furness Council financing costs",
    subtitle="% of net revenue stream. Source: W&F Treasury Management Strategy 2026/27"))

# 3. Unsupported borrowing in Cumberland's capital programme
ub = pd.DataFrame({"year": ["2024/25", "2025/26", "2026/27 (est)"],
                   "borrowing": [117.95, 171.67, 189.67]})
c3 = bar_with_labels(ub, "year", "borrowing",
    "Cumberland Council unsupported borrowing in future capital programme",
    "£m. Source: CIPFA external assurance review (May 2026)")


# 4. Borrowing the investment fund could service
def supported(rate, payment=11.1, years=30):
    return payment * (1 - (1 + rate) ** -years) / rate


cap = pd.DataFrame({"rate": [3, 4, 5, 6, 7]})
cap["supported"] = cap["rate"].apply(lambda r: supported(r / 100))
cap["series"] = "Supportable borrowing"
bars = alt.Chart(cap).mark_bar().encode(
    x=alt.X("rate:O", title="Borrowing rate (%)", axis=alt.Axis(labelAngle=0)),
    y=alt.Y("supported:Q", title="£m", scale=alt.Scale(domain=[0, 350])),
    color=COLOUR)
labels = alt.Chart(cap).mark_text(dy=-6).encode(
    x="rate:O", y="supported:Q", text=alt.Text("supported:Q", format=".0f"))
headline = alt.Chart(pd.DataFrame({"y": [333], "series": ["Headline"]})).mark_rule(
    strokeDash=[4, 4]).encode(y="y:Q", color=COLOUR)
c4 = (bars + labels + headline).properties(title=alt.Title(
    "Borrowing supportable by £11.1m a year over 30 years",
    subtitle="£m; dashed line is the £333m headline. Source: author's calculation; fund size from Cumberland Council papers"))

# 5. External borrowing by body
debt = pd.DataFrame({
    "body": ["Cumberland Council", "Westmorland and Furness Council",
             "Cumbria fire authority", "Cumbria police commissioner", "Combined authority"],
    "borrowing": [282.12, 177.28, 7.59, 0.0, 0.0],
    "series": "main"})
bars = alt.Chart(debt).mark_bar().encode(
    y=alt.Y("body:N", sort="-x", title=None),
    x=alt.X("borrowing:Q", title="£m"),
    color=COLOUR)
labels = alt.Chart(debt).mark_text(dx=4, align="left").encode(
    y=alt.Y("body:N", sort="-x"), x="borrowing:Q",
    text=alt.Text("borrowing:Q", format=".1f"))
c5 = (bars + labels).properties(title=alt.Title(
    "External borrowing by Cumbrian public body",
    subtitle="£m, excluding PFI and leases; dates differ. Sources: CIPFA, W&F, CCFRA, PFCC"))

# 6. Existing debt cost against current PWLB rates
rates = pd.DataFrame({
    "measure": ["W&F existing debt (average)", "PWLB 10-year", "PWLB 25-year", "PWLB 50-year"],
    "rate": [3.70, 5.72, 6.29, 6.08],
    "series": ["Existing debt", "New borrowing", "New borrowing", "New borrowing"]})
bars = alt.Chart(rates).mark_bar().encode(
    x=alt.X("measure:N", sort=None, title=None,
            axis=alt.Axis(labelAngle=0, labelLimit=140)),
    y=alt.Y("rate:Q", title="%"),
    color=COLOUR)
labels = alt.Chart(rates).mark_text(dy=-6).encode(
    x=alt.X("measure:N", sort=None), y="rate:Q", text=alt.Text("rate:Q", format=".2f"))
c6 = (bars + labels).properties(title=alt.Title(
    "Existing debt cost against current PWLB rates",
    subtitle="%, 31 March 2026. Sources: W&F TMSS 2026/27; MUFG via council treasury report"))

for chart, name in [(c1, "cumberland_gap"), (c2, "wf_financing"), (c3, "unsupported_borrowing"),
                    (c4, "fund_capacity"), (c5, "borrowing_by_body"), (c6, "old_vs_new_rates")]:
    styles.save(chart, path=".", name=name, svg=True)